# March 22 — AIC Competition Deep Dive
## Task Parameters · Scoring System · ACT Hyperparams · Running Experiments

> Session notes from thorough codebase analysis (2026-03-22)

---

## 1. Submission Structure — The Most Important Thing

**One Docker image. One policy class. Three trials.**

The same `insert_cable()` method handles all three trials. You never submit different policies per trial. The engine sends a different `Task` message each time and your code reads it.

```python
# The Task message fields your policy receives:
task.cable_type        # e.g. "sfp_sc"
task.cable_name        # e.g. "cable_0"
task.plug_type         # "sfp" or "sc"  <-- branch on this
task.plug_name         # e.g. "sfp_tip" or "sc_tip"
task.port_type         # "sfp" or "sc"
task.port_name         # e.g. "sfp_port_0" or "sc_port_base"
task.target_module_name  # e.g. "nic_card_mount_0" or "sc_port_1"
task.time_limit        # seconds (180 in sample config)
```

**Submission limits:**
- 1 submission per day (team-wide)
- ECR image tags are **immutable** — increment every push (`:v1`, `:v2`, ...)
- Portal login credentials go to team leaders by end of March 2026
- Always test locally first: `docker compose -f docker/docker-compose.yaml up`

---

## 2. Task Parameters — What Gets Randomised

Trials come from **fixed YAML configs** loaded by `aic_engine`. Not randomly generated at runtime. The sample config lives at `aic_engine/config/sample_config.yaml`. The actual eval config is different (not published) but follows the same structure.

### Three trials in qualification:

| Trial | Plug type | Target | What's randomised |
|-------|-----------|--------|---------|
| 1 | SFP module | sfp_port_0 on NIC card | Board pose (x,y,yaw), which nic_rail (0-4) the card is on, card translation on rail, card yaw |
| 2 | SFP module | sfp_port_0 on NIC card | Same as Trial 1 but different random seed |
| 3 | SC plug | sc_port_base on SC port | Board pose (x,y,yaw), SC rail translation |

### What the YAML config controls (per trial):

```yaml
trials:
  trial_1:
    scene:
      task_board:
        pose: {x, y, z, roll, pitch, yaw}   # board position in world
        nic_rail_0:
          entity_present: True/False
          entity_name: "nic_card_0"          # which NIC model to spawn
          entity_pose:
            translation: 0.036              # along rail (m)
            roll/pitch/yaw: 0.0             # orientation offset
        nic_rail_1: {entity_present: False}
        # ... nic_rail_2 through 4
        sc_rail_0: {entity_present: True, entity_pose: {translation, yaw}}
        lc_mount_rail_0/1: {entity_present, entity_pose}
        sfp_mount_rail_0/1: {entity_present, entity_pose}
        sc_mount_rail_0/1: {entity_present, entity_pose}
      cables:
        cable_0:
          attach_cable_to_gripper: True      # robot starts holding plug
          cable_type: "sfp_sc_cable"         # or sfp_sc_cable_reversed
          pose:
            gripper_offset: {x, y, z}        # plug offset in gripper frame
            roll/pitch/yaw: ...              # cable orientation
    tasks:
      task_1:
        cable_type, cable_name, plug_type, plug_name
        port_type, port_name, target_module_name
        time_limit: 180
```

### Rail translation limits (from sample_config.yaml):

```yaml
task_board_limits:
  nic_rail:   {min: -0.0215m, max: 0.0234m}   # ~4.5cm range
  sc_rail:    {min: -0.06m,   max: 0.055m}    # ~11.5cm range
  mount_rail: {min: -0.09425m, max: 0.09425m} # ~18.9cm range
```

### Eval constraints (from qualification_phase.md):
- Roll and pitch of ALL components = **always 0.0** in official eval
- SC port yaw = **always 0.0** in official eval
- Robot starts with plug **already in hand**, close to target
- Grasp pose has small deviations (~2mm, ~0.04 rad) — policy must be robust
- Target port is **always within view** of robot cameras

**Implication:** Your policy doesn't need to handle full workspace search — the port is always visible. The hard part is precise alignment and insertion.

---

## 3. Scoring System — Full Breakdown

### Score breakdown (100 pts max per trial)

**IMPORTANT: `scoring_tests.md` has OUTDATED numbers. Use `scoring.md` as truth.**

| Tier | Category | Range | Condition |
|------|----------|-------|-----------|
| 1 | Model validity | 0 or 1 | Pass/fail: model loads, responds to InsertCable action |
| 2 | Trajectory smoothness | 0–6 | Savitzky-Golay jerk filter; 0 m/s³ → 6pts, ≥50 m/s³ → 0pts. **Only awarded if Tier3 > 0** |
| 2 | Task duration | 0–12 | ≤5s → 12pts, ≥60s → 0pts, linear between. **Only if Tier3 > 0** |
| 2 | Path efficiency | 0–6 | Path ≤ initial plug-port dist → 6pts, ≥1m extra → 0pts. **Only if Tier3 > 0** |
| 2 | Force penalty | 0 to −12 | Force > 20N for > 1s → −12pts |
| 2 | Off-limit contact | 0 to −24 | Any robot link touching enclosure/walls/task_board → −24pts |
| 3 | Correct insertion | 75 | Full insertion into correct port (contact sensors) |
| 3 | Wrong insertion | −12 | Inserted into wrong port |
| 3 | Partial insertion | 38–50 | Plug inside port bounding box (within 5mm x-y tolerance) |
| 3 | Proximity | 0–25 | Distance-based when not inserted (0 at max dist, 25 at port entrance) |

**Max per trial = 100. Max total = 300.**

### Key insight: Tier 2 is gated on Tier 3
Smoothness, duration, and efficiency are **only awarded** if the plug ends up close to or inside the port. Waving the arm smoothly scores nothing on Tier 2. You need proximity first.

### Off-limit models (from WallToucher scoring example):
- `enclosure` — floor, corner posts, ceiling
- `enclosure walls` — transparent acrylic panels
- `task_board` — the board + everything mounted on it (NIC cards, SC ports, etc.)

Only robot links trigger the penalty — the cable itself does not.

### Scoring output format (`scoring.yaml`):
```yaml
trial_1:
  tier1:
    score: 1
    message: "Model validation succeeded."
  tier2:
    score: 20.8
    categories:
      smoothness: {score: 5.2, message: "..."}
      duration:   {score: 10.6, message: "..."}
      efficiency: {score: 5.0, message: "..."}
      force:      {score: 0.0}
      contacts:   {score: 0.0}
  tier3:
    score: 9.9
    message: "Proximity score..."
```

---

## 4. How the Scorer Works Internally

The scorer is a **C++ ROS 2 node** inside `aic_scoring/`. It's not a standalone script — it runs as part of `aic_engine` during trials.

### What it does:
1. **During trial:** `ScoringTier2::StartRecording()` subscribes to all topics and writes a rosbag
2. **After trial:** `ScoringTier2::ComputeScore()` reads the bag offline and calculates metrics
3. Writes `scoring.yaml` to `$AIC_RESULTS_DIR`

### Topics recorded for scoring:
```
/joint_states                         # robot joint positions
/tf                                   # gripper pose
/tf_static                            # static transforms
/scoring/tf                           # cable plug+port poses (ground truth internal)
/aic/gazebo/contacts/off_limit        # collision events
/fts_broadcaster/wrench               # force-torque sensor
/aic_controller/joint_commands        # your joint commands
/aic_controller/pose_commands         # your Cartesian commands
/scoring/insertion_event              # plug-port contact event
/aic_controller/controller_state      # TCP pose, velocities
```

### Can we use the scorer locally?

**Yes — it runs automatically inside the Docker eval container.** Every `docker run ... aic_eval:latest` run with `start_aic_engine:=true` runs the scorer and writes `scoring.yaml` to `$AIC_RESULTS_DIR`.

You can also run from source (if you build `~/ws_aic` natively) using the three-terminal pattern from `scoring_tests.md`. Docker is simpler and already set up.

**You CANNOT run the scorer standalone** on an arbitrary rosbag — it's integrated into the engine lifecycle. The scorer starts/stops recording based on engine state machine transitions.

---

## 5. How to Run All Experiments

### Our setup (Docker-based, already working)

```bash
# Terminal 1: Eval environment
POLICY_NAME=CheatCode  GT=true   # or: WaveArm/false, GentleGiant/false, etc.
RUN_DIR=~/projects/Project-Automaton/aic_results/${POLICY_NAME}_$(date +%Y%m%d_%H%M%S)
mkdir -p "$RUN_DIR"

docker run -it --rm \
  --name aic_eval --network host --gpus all \
  -e DISPLAY=:0 -e WAYLAND_DISPLAY=wayland-0 \
  -e XDG_RUNTIME_DIR=/mnt/wslg/runtime-dir \
  -e PULSE_SERVER=/mnt/wslg/PulseServer \
  -e GALLIUM_DRIVER=d3d12 \
  -e LD_LIBRARY_PATH=/usr/lib/wsl/lib \
  -e AIC_RESULTS_DIR=/aic_results \
  -v /tmp/.X11-unix:/tmp/.X11-unix -v /mnt/wslg:/mnt/wslg \
  -v /usr/lib/wsl:/usr/lib/wsl -v "$RUN_DIR":/aic_results \
  --device /dev/dxg \
  ghcr.io/intrinsic-dev/aic/aic_eval:latest \
  ground_truth:=$GT start_aic_engine:=true

# Terminal 2: Policy (wait for "Retrying..." first)
cd ~/projects/Project-Automaton/References/aic
pixi run ros2 run aic_model aic_model \
  --ros-args -p use_sim_time:=true \
  -p policy:=aic_example_policies.ros.$POLICY_NAME
```

### All example policies and what they test:

| Policy | GT? | Expected Tier2 | Expected Tier3 | Purpose |
|--------|-----|----------------|----------------|---------|
| WaveArm | No | smoothness only if near port | proximity | API skeleton baseline |
| CheatCode | **Yes** | high smoothness + duration + efficiency | 75 (full insert) | Gold standard to beat |
| GentleGiant | No | 0 (plug not near port) | 0 | Low jerk reference |
| SpeedDemon | No | −12 force penalty | 0 | High jerk reference |
| WallToucher | No | −24 contact penalty | 0 | Off-limit contact demo |
| WallPresser | No | −12 force + −24 contact | 0 | Excessive force demo |
| RunACT | No | varies | varies | Neural net baseline |

### Alternative: Build from source (no Docker)
From `scoring_tests.md` — if you build `~/ws_aic` natively, use THREE terminals:
```bash
# Terminal 0
source ~/ws_aic/install/setup.bash
export RMW_IMPLEMENTATION=rmw_zenoh_cpp
ros2 run rmw_zenoh_cpp rmw_zenohd

# Terminal 1
source ~/ws_aic/install/setup.bash
ros2 run aic_model aic_model --ros-args -p use_sim_time:=true -p policy:=...

# Terminal 2
AIC_RESULTS_DIR=~/aic_results/run_name \
ros2 launch aic_bringup aic_gz_bringup.launch.py \
  ground_truth:=true start_aic_engine:=true
```
Our Docker approach already handles Terminals 0+2 inside the container.

---

## 6. Teleoperation & Data Collection Pipeline

Yes — fully supported via `aic_utils/lerobot_robot_aic/`.

### Three teleop modes:
```bash
cd ~/projects/Project-Automaton/References/aic

# Keyboard cartesian (recommended)
pixi run lerobot-teleoperate \
  --robot.type=aic_controller --robot.id=aic \
  --teleop.type=aic_keyboard_ee --teleop.id=aic \
  --robot.teleop_target_mode=cartesian \
  --robot.teleop_frame_id=base_link \
  --display_data=true

# SpaceMouse cartesian
# --teleop.type=aic_spacemouse

# Joint-space keyboard
# --teleop.type=aic_keyboard_joint --robot.teleop_target_mode=joint
```

### Keyboard map (cartesian):
| Key | Motion | Key | Motion |
|-----|--------|-----|--------|
| w/s | ±linear y | r/f | ±linear z |
| a/d | ±linear x | q/e | ±angular z |
| Shift+w/s | ±angular x | Shift+a/d | ±angular y |
| t | toggle slow/fast | | |

### Record demos:
```bash
pixi run lerobot-record \
  --robot.type=aic_controller --robot.id=aic \
  --teleop.type=aic_keyboard_ee --teleop.id=aic \
  --robot.teleop_target_mode=cartesian --robot.teleop_frame_id=base_link \
  --dataset.repo_id=${HF_USER}/aic_demo_v0 \
  --dataset.single_task=aic_insert \
  --dataset.push_to_hub=false --dataset.private=true \
  --display_data=true
# Right Arrow = next episode, Left Arrow = re-record, ESC = stop
```

### Train:
```bash
pixi run lerobot-train \
  --dataset.repo_id=${HF_USER}/aic_demo_v0 \
  --policy.type=act \
  --output_dir=outputs/train/act_v0 \
  --policy.device=cuda \
  --policy.repo_id=${HF_USER}/my_act_policy
```

### Three data sources to build:
| Source | How | Strength |
|--------|-----|----------|
| CheatCode + `ground_truth:=true` | Auto-runs all trials, saves rosbags | Fast GT demos at scale, perfect insertions |
| `lerobot-record` keyboard teleop | Manual in Gazebo | High-quality human demos, realistic approach strategies |
| Isaac Lab `record_demos.py` | Teleop in Isaac | Sim diversity, domain randomisation |

**Pro tip:** Use CheatCode to collect hundreds of GT demo trajectories first. These are essentially perfect demonstrations you can use as imitation learning targets.

---

## 7. ACT (RunACT) — Hyperparameters & What to Change

### Current RunACT.py — what it does
- Downloads `grkw/aic_act_policy` from HuggingFace
- Uses 3 wrist camera images + 26-dim robot state as input
- Outputs 7-dim velocity action (6 Cartesian DOF + gripper)
- Runs at ~4Hz for 30 seconds
- **Ignores `task.plug_type` entirely** — same behaviour for all trials

### Input state vector (26 dims) — order must match training:
```
TCP position (3):    tcp_pose.position.x/y/z
TCP orientation (4): tcp_pose.orientation.x/y/z/w  (quaternion)
TCP linear vel (3):  tcp_velocity.linear.x/y/z
TCP angular vel (3): tcp_velocity.angular.x/y/z
TCP error (6):       controller_state.tcp_error[0:6]
Joint positions (7): joint_states.position[0:7]
```

### Image preprocessing:
```python
image_scaling = 0.25   # resize to 25% of raw resolution
# Then per-camera normalise: (pixel/255 - mean) / std
# Stats loaded from policy_preprocessor_step_3_normalizer_processor.safetensors
```

### Parameters to customise if retraining:

| Parameter | Location in RunACT.py | Current value | Notes |
|-----------|----------------------|---------------|-------|
| `repo_id` | line 62 | `"grkw/aic_act_policy"` | Change to your HF repo |
| `image_scaling` | line 131 | `0.25` | **Must match training** |
| State vector order | `prepare_observations()` lines 202–227 | 26-dim as above | Must exactly match training data |
| Inference duration | line 251 | `30.0` seconds | Tune per your policy |
| Control rate sleep | line 292 | `0.25s` (~4Hz) | Interacts with ACT chunk size |
| Normalization stats | `.safetensors` file | Baked into checkpoint | Auto-loaded from checkpoint |

### ACT training hyperparameters (LeRobot defaults for ACT):
```bash
pixi run lerobot-train \
  --policy.type=act \
  --policy.chunk_size=100          # action chunk length (key ACT hyperparam)
  --policy.n_action_steps=100      # how many steps to execute per inference
  --policy.hidden_dim=512          # transformer hidden dim
  --policy.dim_feedforward=3200    # FFN width
  --policy.n_heads=8               # attention heads
  --policy.n_encoder_layers=4      # encoder depth
  --policy.n_decoder_layers=7      # decoder depth
  --policy.dropout=0.1             # regularisation
  --policy.kl_weight=10.0          # CVAE KL loss weight
  --training.batch_size=8          # keep small if memory limited
  --training.num_workers=4         # dataloader workers
  --training.lr=1e-5               # learning rate
  --training.num_epochs=100        # train longer for better results
```

### Critical gap to fix in RunACT:
Current RunACT **ignores the Task message**. For a real submission either:
1. Branch on `task.plug_type` and load different checkpoints (SFP vs SC)
2. Train a **single unified policy** that generalises across both from vision

Option 2 is harder but only needs one checkpoint. Option 1 is easier but requires two trained models.

### Cloud evaluator memory constraint:
Eval runs on **NVIDIA L4 (24 GB VRAM)**. Your 5090 has 32 GB. Check before submitting:
```python
total = sum(p.numel() for p in self.policy.parameters())
print(f"Model params: {total/1e6:.1f}M")
# ACT at default size ~80-120M params — should be fine on L4
```

---

## 8. Policy Design Notes — Per Policy

| Policy | GT needed? | Eval-legal? | Tier 2 expected | Tier 3 expected | Key gotcha |
|--------|-----------|-------------|-----------------|-----------------|------------|
| WaveArm | No | Yes | ~21 (proximity luck) | ~10-19 (proximity) | Doesn't attempt insertion |
| CheatCode | **Yes** | **No** | High (20+) | 75 (full insert) | Uses `/scoring/tf` — blocked by Zenoh ACL in eval |
| GentleGiant | No | Yes | 0 (plug not near port) | 0 | Doesn't attempt insertion |
| SpeedDemon | No | Yes | −12 force penalty | 0 | Oscillates violently |
| WallToucher | No | Yes | −24 contact penalty | 0 | Arm extends into enclosure |
| WallPresser | No | Yes | −12 force + −24 contact | 0 | Presses arm into wall |
| RunACT | No | Yes | varies | varies | Ignores task.plug_type, 30s hardcoded |

### CheatCode strategy (the gold standard to replicate without GT):
1. **Phase 1 — Approach (5s):** 100 steps × 50ms, interpolate to 20cm above port + orient plug via quaternion slerp
2. **Phase 2 — Descent:** Lower 0.5mm per step, PI controller corrects XY drift
3. **Stop** when z_offset < −15mm (past port surface)
4. **Hold 5s** to stabilise

To replicate without GT: replace `tf_buffer.lookup_transform("base_link", port_frame)` with **camera-based port detection** outputting the same (x, y, z) target.

### Control mode trade-offs:

| Mode | Stiffness | Damping | Character |
|------|-----------|---------|----------|
| GentleGiant | 50 | 40 | Slow, smooth, low jerk — good for Tier 2 smoothness |
| Default | 90/50 | 50/20 | Balanced |
| SpeedDemon | 500 | 5 | Fast, oscillates, triggers force penalty |
| Insertion phase | 30/20 | 20/10 | Compliant — let contact forces guide the plug |

---

## 9. Observation API — What Your Policy Actually Receives

```python
obs = get_observation()  # up to 20Hz, time-synchronised

# Three wrist cameras (Basler)
obs.left_image    # sensor_msgs/Image
obs.center_image
obs.right_image
obs.left_camera_info   # sensor_msgs/CameraInfo (intrinsics, distortion)
obs.center_camera_info
obs.right_camera_info

# Force/Torque (Axia80 M20, 6-axis, tared at startup ~= 0N baseline)
obs.wrist_wrench.wrench.force.x/y/z    # Newtons
obs.wrist_wrench.wrench.torque.x/y/z   # Nm

# Joint state (6 arm + gripper = 7)
obs.joint_states.name        # ['shoulder_pan_joint', ...]
obs.joint_states.position    # radians
obs.joint_states.velocity    # rad/s

# Controller state
obs.controller_state.tcp_pose         # current gripper TCP Pose
obs.controller_state.tcp_velocity     # Twist (linear + angular)
obs.controller_state.tcp_error        # 6-element position/orientation error
```

**What you do NOT get in eval:**
- Ground truth TF of port/plug positions (blocked by Zenoh ACL)
- Depth images (no official depth stream)
- Access to `/gazebo`, `/gz_server`, `/scoring` topics

---

## 10. Next Steps Priority Order

1. **Run CheatCode** with `ground_truth:=true` — get the 75pt baseline score + collect demo data
2. **Run remaining example policies** (GentleGiant, SpeedDemon, WallToucher, WallPresser) to understand scoring edge cases
3. **Build `collect_scores.py`** — parse all `scoring.yaml` files into one CSV for comparison
4. **Data collection sprint** — use CheatCode + lerobot-record to build a demo dataset
5. **Retrain ACT** on your own data, fix the Task message branching (SFP vs SC)
6. **Replace CheatCode's GT lookup** with camera-based perception — first eval-legal insertion attempt

**Key milestone:** First eval-legal policy that earns Tier 3 > 0 in Gazebo (gets near/into the port without ground truth).